In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def run_bootstrap_simulation(data_files, n_samples=10000, sample_size=40):
    """
    Performs a bootstrapping simulation on NHL game data to calculate GF%.

    Args:
        data_files (list of str): A list of file paths to the CSV data files,
                                  where each file represents a single NHL season
                                  and contains a 'SEASON' column.
        n_samples (int): The number of bootstrap samples to generate for each team.
        sample_size (int): The number of games to randomly sample for each bootstrap.

    Returns:
        pandas.DataFrame: A DataFrame containing the calculated GF% for each bootstrap
                          sample, along with the corresponding team and season.
    """
    # Initialize a list to hold all the results before concatenating into a single DataFrame.
    # This is more memory-efficient than appending to a DataFrame in a loop.
    all_bootstrap_samples = []

    print(f"Starting the bootstrapping simulation for {len(data_files)} seasons...")

    # Outer loop: Iterate through each data file (season)
    for file_path in data_files:
        try:
            # Read the CSV data into a pandas DataFrame
            df = pd.read_csv(file_path)

            # Check for the required 'SEASON' column
            if 'SEASON' not in df.columns:
                print(f"Error: Skipping file {file_path}. The 'SEASON' column is missing.")
                continue

            # Get the season name directly from the DataFrame
            season = df['SEASON'].iloc[0]
            print(f"\nProcessing season: {season} from file: {file_path}")

            # Get the unique teams for the current season
            teams = df['TEAM'].unique()

            # Inner loop: Iterate through each team in the current season
            for team in teams:
                # Get all games played by the current team
                team_data = df[df['TEAM'] == team]

                # Check if the team has enough games to sample from
                if len(team_data) < sample_size:
                    print(f"  Skipping team {team} for season {season}: Not enough games ({len(team_data)}).")
                    continue

                # Innermost loop: Perform the bootstrapping simulation
                # This loop runs 'n_samples' times for each team-season
                team_bootstrap_results = []
                for i in range(n_samples):
                    # Randomly sample 'sample_size' games with replacement
                    # The 'replace=True' argument enables bootstrapping with replacement,
                    # which is the standard method for this statistical technique.
                    sampled_games = team_data.sample(n=sample_size, replace=True, random_state=i)

                    # Calculate the total Goals For and Goals Against from the sample
                    total_gf = sampled_games['GF'].sum()
                    total_ga = sampled_games['GA'].sum()

                    # Calculate the Goals For Percentage (GF%)
                    # Handle the case where total_gf + total_ga is 0 to avoid division by zero
                    if (total_gf + total_ga) == 0:
                        gf_percent = 0.0
                    else:
                        gf_percent = total_gf / (total_gf + total_ga)

                    # Store the result for this bootstrap sample
                    team_bootstrap_results.append({
                        'season': season,
                        'team': team,
                        'gf_percent': gf_percent,
                        'sample_number': i + 1
                    })

                # Convert the list of results for the current team into a DataFrame
                team_df = pd.DataFrame(team_bootstrap_results)

                # Append the team's results to the overall list
                all_bootstrap_samples.append(team_df)
                
                print(f"  Completed bootstrapping for team {team} ({n_samples} samples).")

        except FileNotFoundError:
            print(f"Error: File not found at {file_path}. Skipping.")
        except Exception as e:
            print(f"An error occurred while processing {file_path}: {e}")

    # Concatenate all the individual team DataFrames into a single, large DataFrame
    if all_bootstrap_samples:
        final_df = pd.concat(all_bootstrap_samples, ignore_index=True)
        print("\nBootstrapping simulation complete. Final DataFrame generated.")
        return final_df
    else:
        print("\nNo data processed. Returning empty DataFrame.")
        return pd.DataFrame()

In [ ]:
scraped_data_files = ['scraping_talentdist_200506.csv']